In [ ]:
from fastapi import FastAPI, UploadFile, File
from fastapi.middleware.cors import CORSMiddleware
import pandas as pd
import joblib
import uvicorn
import io
import nest_asyncio
import threading
import time

app = FastAPI(title="Student Placement Prediction API")
model = joblib.load("xgboost_placement_model.pkl")

# Enable CORS so your index.html can talk to this API
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Load the saved model pipeline and preprocessor
try:
    # This loads the XGBoost model
    model = joblib.load("xgboost_placement_model.pkl")
    # This loads the Scaler and Encoder
    preprocessor = joblib.load("preprocessor.pkl")
    print("✅ Model and Preprocessor loaded successfully.")
except Exception as e:
    print(f"❌ Error loading files: {e}")
    model = None
    preprocessor = None

@app.post("/predict")
async def predict(data: dict):
    if model is None or preprocessor is None:
        return {"error": "Model files missing. Please run student_pp.py first."}

    # 1. Convert JSON data to DataFrame
    df = pd.DataFrame([data])

    # 2. FEATURE ENGINEERING (Must match student_pp.py exactly)
    # Adding the interaction terms and composite score used during training
    df['composite_score'] = (
        df['cgpa'] * 10 +
        df['internships'] * 15 +
        df['projects'] * 2 +
        df['aptitude_score'] * 0.3 +
        df['communication_score'] * 2 -
        df.get('active_backlogs', 0) * 18
    )
    df['cgpa_x_internship'] = df['cgpa'] * df['internships']
    df['aptitude_x_projects'] = df['aptitude_score'] * df['projects']

    # Ensure categorical columns exist (use defaults if missing from request)
    if 'department' not in df: df['department'] = 'CSE'
    if 'skill_stack' not in df: df['skill_stack'] = 'Data Science'
    if 'active_backlogs' not in df: df['active_backlogs'] = 0

    # 3. PREPROCESSING
    # Transform the raw data using the saved preprocessor (StandardScaler + OneHot)
    try:
        processed_data = preprocessor.transform(df)
    except Exception as e:
        return {"error": f"Preprocessing failed: {str(e)}"}

    # 4. PREDICTION
    prediction = model.predict(processed_data)[0]
    probability = model.predict_proba(processed_data)[0][1]

    return {
        "placement_status": "Placed" if int(prediction) == 1 else "Not Placed",
        "placement_probability": f"{round(float(probability) * 100, 2)}%",
        "recommendation": "High chance! Keep up the technical projects." if prediction == 1 else "Focus on improving CGPA and Aptitude scores."
    }

@app.post("/resume-analyzer")
async def analyze_resume(file: UploadFile = File(...)):
    content = await file.read()
    text = content.decode("utf-8", errors="ignore")

    # Simple keyword matching for resume scoring
    keywords = ["python", "machine learning", "data science", "sql", "java", "aws", "internship"]
    found = [word for word in keywords if word.lower() in text.lower()]

    score = round((len(found) / len(keywords)) * 100, 2)

    return {
        "resume_score_percentage": score,
        "keywords_found": found,
        "suggestion": "Include more specific technical stack mentions like 'FastAPI' or 'XGBoost'."
    }

def run_uvicorn():
    nest_asyncio.apply() # Apply nest_asyncio to allow nested event loops within the thread
    uvicorn.run(app, host="0.0.0.0", port=8000)

# Start Uvicorn in a separate thread
if __name__ == "__main__":
    thread = threading.Thread(target=run_uvicorn)
    thread.start()
    print("Uvicorn server started in a background thread on http://0.0.0.0:8000")
    print("You can now make requests to the API.")
    # Optionally, keep the main thread alive for a bit or indefinitely if needed
    # while True:
    #     time.sleep(1)
    # Load model and preprocessor
try:
    model = joblib.load("xgboost_placement_model.pkl")
    preprocessor = joblib.load("preprocessor.pkl")
except:
    model = None
    preprocessor = None


@app.post("/predict")
async def predict(data: dict):
    if model is None or preprocessor is None:
        return {"error": "Model or Preprocessor file not found."}

    df = pd.DataFrame([data])

    # Apply preprocessing before prediction
    processed_data = preprocessor.transform(df)

    prediction = model.predict(processed_data)[0]
    probability = model.predict_proba(processed_data)[0][1]












    return {
        "placement_status": int(prediction),
        "placement_probability": float(round(probability, 3))
    }
model = joblib.load("xgboost_placement_model.pkl")
preprocessor = joblib.load("preprocessor.pkl")    


✅ Model and Preprocessor loaded successfully.
Uvicorn server started in a background thread on http://0.0.0.0:8000
You can now make requests to the API.


INFO:     Started server process [36776]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 8000): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
